### Training for with embeddings using XGBoost

In [9]:
import pandas as pd
import numpy as np
import joblib
import os
from google.colab import drive
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# 1. Load Data (Assuming Drive is mounted)
DATA_PATH = '/content/drive/My Drive/Colab Notebooks/project data/'
df = pd.read_csv(os.path.join(DATA_PATH, 'Twitter_Analysis.csv'))
embeddings = np.load(os.path.join(DATA_PATH, 'embeddings_384.npy')).astype('float32')

# 2. Prepare X and y
embedding_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
df_embeddings = pd.DataFrame(embeddings, columns=embedding_cols, index=df.index)

COLS_TO_DROP = ["embeddings",          # undocumented, not in official dataset
    "tweet",               # raw text, not a feature
    "statement",           # raw text, not a feature  
    "majority_target",     # string version of label
    "BinaryNumTarget",     # this is your y, not a feature
    'followers_count', 'friends_count', 'favourites_count', # (User favourites)
    'statuses_count', 'listed_count', 'following', 
    'BotScore', 'BotScoreBinary', 'cred', 'normalize_influence',
    'retweets',
    'quotes', 'Unnamed: 0']
# Add your other metadata drops here...

X = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])
X = pd.concat([X, df_embeddings], axis=1)
y = df["BinaryNumTarget"].astype(int)

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
# 3. XGBoost GPU Configuration
# 'tree_method': 'gpu_hist' is the magic command for Colab GPUs
model = XGBClassifier(
    tree_method='hist', 
    device='cuda',
    random_state=42,
    eval_metric='logloss'
)

# 4. GridSearch Parameters
param_grid = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [200, 400],
    'max_depth': [6, 10, 12],       # XGBoost depth is usually smaller than LGBM leaves
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    #'gamma': [0, 0.1]               # Minimum loss reduction to split
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Starting GPU-accelerated GridSearch...")
search = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    verbose=2,
    n_jobs=1 # Keep this 1; the GPU handles the internal parallelism
)

search.fit(X_tr, y_tr)

Starting GPU-accelerated GridSearch...
Fitting 3 folds for each of 48 candidates, totalling 144 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=  16.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=  15.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=  15.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  15.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  15.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  15.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=  18.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsa

In [ ]:
# 5. Results
print(f"Best Params: {search.best_params_}")
best_model = search.best_estimator_
print(classification_report(y_val, best_model.predict(X_val)))

# Save
joblib.dump(best_model, os.path.join(DATA_PATH, 'social_model_xgboost_gpu.joblib'))

Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 400, 'subsample': 0.8}
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13054
           1       1.00      1.00      1.00     13786

    accuracy                           1.00     26840
   macro avg       1.00      1.00      1.00     26840
weighted avg       1.00      1.00      1.00     26840



['/content/drive/My Drive/Colab Notebooks/project data/social_model_xgboost_gpu.joblib']

In [7]:
import matplotlib.pyplot as plt

# Get feature importances
importances = best_model.feature_importances_
feature_names = X_tr.columns
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

# Print the top 10
print(feature_importance_df.head(10))

             feature  importance
0         Unnamed: 0    0.155329
398          emb_339    0.039292
170          emb_111    0.032085
385          emb_326    0.026073
379          emb_320    0.024985
152           emb_93    0.023219
309          emb_250    0.021228
277          emb_218    0.018452
25   DATE_percentage    0.013720
161          emb_102    0.013565
